# Nsight Compute Structured Review

This notebook reads the SQLite-first analysis bundle generated by `scripts/analyze_ncu_run.py`.
It focuses on the structured core section imports rather than string-hit heuristics so we can review one `ncu` run in a way that is much closer to the Nsight Compute UI.


In [ ]:
from pathlib import Path

RUN_ID = "20260319-1514-train-completion-01"
WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (
        path
        for path in [WORKING_DIR, *WORKING_DIR.parents]
        if (path / "profiling").exists() and (path / "artifacts").exists()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the Griffin repo root from the current working directory.")

ANALYSIS_DIR = REPO_ROOT / "artifacts" / "profiles" / "analysis" / RUN_ID
DB_PATH = ANALYSIS_DIR / "ncu_analysis.sqlite"


In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from textwrap import dedent

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DB_PATH}. Open this notebook from the Griffin repo or rerun the setup cell so REPO_ROOT resolves correctly."
    )

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda value: f"{value:0.3f}")

conn = sqlite3.connect(DB_PATH)
SUCCESS_STATUSES = {"success", "cached_success"}

def q(sql: str, params=None) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params or [])

def display_md(text: str) -> None:
    display(Markdown(dedent(text).strip()))

def metric_summaries(section_ids) -> pd.DataFrame:
    placeholders = ", ".join(["?"] * len(section_ids))
    sql = f"""
        SELECT
            section_id,
            metric_name,
            metric_unit,
            value_count,
            ROUND(min_value, 3) AS min_value,
            ROUND(avg_value, 3) AS avg_value,
            ROUND(max_value, 3) AS max_value
        FROM numeric_metric_summaries
        WHERE section_id IN ({placeholders})
        ORDER BY section_id, metric_name
    """
    return q(sql, section_ids)

def metric_value(summary_df: pd.DataFrame, section_id: str, metric_name: str) -> float:
    match = summary_df[
        (summary_df["section_id"] == section_id) & (summary_df["metric_name"] == metric_name)
    ]
    if match.empty:
        return np.nan
    return float(match.iloc[0]["avg_value"])

def sampled_metrics(pairs) -> pd.DataFrame:
    if not pairs:
        return pd.DataFrame(
            columns=[
                "sample_rank",
                "launch_id",
                "section_id",
                "metric_name",
                "metric_unit",
                "metric_value_num",
            ]
        )
    clauses = " OR ".join(["(section_id = ? AND metric_name = ?)"] * len(pairs))
    params = [item for pair in pairs for item in pair]
    sql = f"""
        SELECT
            sample_rank,
            launch_id,
            section_id,
            metric_name,
            metric_unit,
            metric_value_num
        FROM sampled_section_metric_values
        WHERE metric_value_num IS NOT NULL
          AND ({clauses})
        ORDER BY sample_rank, section_id, metric_name
    """
    return q(sql, params)

def sampled_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    summary = (
        df.groupby(["section_id", "metric_name", "metric_unit"], dropna=False)["metric_value_num"]
        .agg(samples="size", min_value="min", median_value="median", avg_value="mean", max_value="max")
        .reset_index()
    )
    return summary.round(3)

def plot_bar(ax, data: pd.Series, title: str, ylabel: str, color: str = "#4C72B0") -> None:
    data.plot.bar(ax=ax, legend=False, color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)

def pct(value: float) -> str:
    if pd.isna(value):
        return "n/a"
    return f"{value:0.2f}%"

def num(value: float, digits: int = 3) -> str:
    if pd.isna(value):
        return "n/a"
    return f"{value:0.{digits}f}"


def summary_rows(summary_df: pd.DataFrame, patterns) -> pd.DataFrame:
    if summary_df.empty:
        return summary_df
    mask = summary_df["metric_name"].fillna("").apply(
        lambda metric_name: any(pattern in metric_name for pattern in patterns)
    )
    return summary_df[mask].reset_index(drop=True)


def ratio_pct(numerator: float, denominator: float) -> float:
    if pd.isna(numerator) or pd.isna(denominator) or denominator == 0:
        return np.nan
    return 100.0 * numerator / denominator


## Notebook Upgrade Checklist

- [x] Capture health with per-section import coverage, timing, and reuse state.
- [x] Kernel identity and match scope including sampled-launch coverage.
- [x] Launch Statistics and Occupancy summaries from structured section tables.
- [x] Compute vs memory and Speed-of-Light summaries from structured section tables.
- [x] Structured WorkloadDistribution run-level and sampled views.
- [x] Scheduler and Warp State summaries from structured section tables.
- [x] Sampled per-launch distributions across the repeated kernel launches.
- [x] Structured Nsight-style rule/advice cards sourced from section rows.
- [x] Guided interpretation and explicit coverage-gap reporting.
- [ ] Add source/instruction-level views only if a concrete follow-on investigation requires them.


## Capture Health


In [ ]:
bundle = q("SELECT key, value FROM bundle_metadata ORDER BY key")
bundle_map = dict(zip(bundle["key"], bundle["value"])) if not bundle.empty else {}

session_attempts = q(
    """
    SELECT
        page,
        status,
        timeout_sec,
        ROUND(elapsed_sec, 3) AS elapsed_sec,
        line_count,
        row_count,
        reused_existing
    FROM import_attempts
    ORDER BY page
    """
)
section_attempts = q(
    """
    SELECT
        section_id,
        display_name,
        status,
        timeout_sec,
        ROUND(elapsed_sec, 3) AS elapsed_sec,
        row_count,
        reused_existing
    FROM section_import_attempts
    ORDER BY section_id
    """
)
warnings_df = q("SELECT message FROM extraction_warnings")
launch_scope = q(
    """
    SELECT
        COUNT(DISTINCT launch_id) AS matched_launches,
        MIN(launch_id) AS min_launch_id,
        MAX(launch_id) AS max_launch_id
    FROM section_metric_values
    WHERE launch_id IS NOT NULL
    """
)
sample_scope = q(
    """
    SELECT
        COUNT(*) AS sampled_launches,
        MIN(launch_id) AS sampled_min_launch_id,
        MAX(launch_id) AS sampled_max_launch_id
    FROM sampled_launches
    """
)

matched_launches = int(launch_scope.iloc[0]["matched_launches"] or 0)
sampled_launches = int(sample_scope.iloc[0]["sampled_launches"] or 0)
sample_pct = (100.0 * sampled_launches / matched_launches) if matched_launches else 0.0
completed_sections = int(section_attempts["status"].isin(SUCCESS_STATUSES).sum())
reused_sections = int(section_attempts["reused_existing"].fillna(0).sum())

display_md(
    f"""
    ### Bundle Summary
    - Run ID: `{RUN_ID}`
    - Report path: `{bundle_map.get('rep_path', 'unknown')}`
    - Bundle path: `{ANALYSIS_DIR}`
    - Sections complete: `{completed_sections}/{len(section_attempts)}`
    - Reused completed section sidecars on this run: `{reused_sections}`
    - Matched launches in structured bundle: `{matched_launches}`
    - Sampled launches: `{sampled_launches}` (`{sample_pct:0.2f}%` of matched launches)
    """
)

display_md("### Bundle Metadata")
display(bundle)

display_md("### Section Import Attempts")
display(section_attempts)

display_md("### Session Import Attempts")
display(session_attempts)

if not section_attempts.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    section_attempts.set_index("section_id")["elapsed_sec"].plot.bar(ax=ax, color="#55A868")
    ax.set_title("Core Section Import Time")
    ax.set_ylabel("Elapsed seconds")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()

if warnings_df.empty:
    display_md("""### Extraction Warnings
No extraction warnings were recorded for this bundle.""")
else:
    display_md("### Extraction Warnings")
    display(warnings_df)


## Kernel Identity and Match Scope


In [ ]:
kernel_df = q(
    """
    SELECT kernel_filter, kernel_name, matched_in_strings, string_occurrences
    FROM kernel_targets
    """
)
identity_df = q(
    """
    SELECT
        process_name,
        device,
        cc,
        COUNT(DISTINCT launch_id) AS launches
    FROM section_metric_values
    WHERE kernel_name IS NOT NULL AND TRIM(kernel_name) != ''
    GROUP BY process_name, device, cc
    ORDER BY launches DESC, process_name
    LIMIT 5
    """
)

kernel_name = kernel_df.iloc[0]["kernel_name"] if not kernel_df.empty else "unknown"
kernel_filter = kernel_df.iloc[0]["kernel_filter"] if not kernel_df.empty else "unknown"
min_launch_id = int(launch_scope.iloc[0]["min_launch_id"]) if matched_launches else 0
max_launch_id = int(launch_scope.iloc[0]["max_launch_id"]) if matched_launches else 0
sampled_min = int(sample_scope.iloc[0]["sampled_min_launch_id"]) if sampled_launches else 0
sampled_max = int(sample_scope.iloc[0]["sampled_max_launch_id"]) if sampled_launches else 0

display_md(
    f"""
    ### Match Scope
    - Target kernel: `{kernel_name}`
    - Recorded kernel filter: `{kernel_filter}`
    - Structured launch coverage: `{matched_launches}` launches spanning IDs `{min_launch_id}` to `{max_launch_id}`
    - Sampled launch coverage: `{sampled_launches}` launches spanning IDs `{sampled_min}` to `{sampled_max}`
    - Sample policy: evenly distributed over observed launch order; the sample is for compact visualization, while run-level averages come from the full structured section imports.
    """
)

display_md("### Kernel Target Table")
display(kernel_df)

display_md("### Process / Device Identity")
display(identity_df)


## Launch and Occupancy


In [ ]:
launch_summary = metric_summaries(["LaunchStats"])
occupancy_summary = metric_summaries(["Occupancy"])

launch_focus = launch_summary[
    launch_summary["metric_name"].isin(
        [
            "# SMs",
            "Block Size",
            "Grid Size",
            "Registers Per Thread",
            "Static Shared Memory Per Block",
            "Driver Shared Memory Per Block",
            "Waves Per SM",
        ]
    )
].reset_index(drop=True)
occupancy_focus = occupancy_summary[
    occupancy_summary["metric_name"].isin(
        [
            "Achieved Occupancy",
            "Theoretical Occupancy",
            "Achieved Active Warps Per SM",
            "Theoretical Active Warps per SM",
        ]
    )
].reset_index(drop=True)
occupancy_limiters = occupancy_summary[
    occupancy_summary["metric_name"].isin(
        [
            "Block Limit Registers",
            "Block Limit Shared Mem",
            "Block Limit Warps",
            "Block Limit SM",
        ]
    )
].reset_index(drop=True)

display_md("### Launch Statistics")
display(launch_focus)

display_md("### Occupancy Summary")
display(occupancy_focus)

display_md("### Occupancy Limiters")
display(occupancy_limiters)

launch_occ_summary = pd.concat([launch_summary, occupancy_summary], ignore_index=True)
display_md(
    f"""
    ### Launch / Occupancy Readout
    - Average `Waves Per SM`: `{num(metric_value(launch_occ_summary, 'LaunchStats', 'Waves Per SM'))}`
    - Achieved occupancy: `{pct(metric_value(launch_occ_summary, 'Occupancy', 'Achieved Occupancy'))}`
    - Theoretical occupancy: `{pct(metric_value(launch_occ_summary, 'Occupancy', 'Theoretical Occupancy'))}`
    - Achieved active warps per SM: `{num(metric_value(launch_occ_summary, 'Occupancy', 'Achieved Active Warps Per SM'))}`
    - Theoretical active warps per SM: `{num(metric_value(launch_occ_summary, 'Occupancy', 'Theoretical Active Warps per SM'))}`
    """
)


In [ ]:
occ_pct = pd.Series(
    {
        "Achieved Occupancy": metric_value(occupancy_summary, "Occupancy", "Achieved Occupancy"),
        "Theoretical Occupancy": metric_value(occupancy_summary, "Occupancy", "Theoretical Occupancy"),
    }
)
limiter_series = occupancy_limiters.set_index("metric_name")["avg_value"]

sampled_launch_occ = sampled_metrics(
    [
        ("LaunchStats", "Waves Per SM"),
        ("Occupancy", "Achieved Occupancy"),
        ("Occupancy", "Theoretical Occupancy"),
    ]
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_bar(axes[0], occ_pct, "Occupancy vs Theoretical Ceiling", "Percent")
plot_bar(axes[1], limiter_series, "Occupancy Limiting Factors", "Blocks")

sampled_pivot = sampled_launch_occ.pivot_table(
    index="sample_rank",
    columns=["section_id", "metric_name"],
    values="metric_value_num",
    aggfunc="mean",
)
if ("LaunchStats", "Waves Per SM") in sampled_pivot.columns:
    sampled_pivot[("LaunchStats", "Waves Per SM")].plot(ax=axes[2], color="#C44E52")
    axes[2].set_title("Sampled Waves Per SM")
    axes[2].set_ylabel("Waves")
    axes[2].set_xlabel("Sample Rank")
plt.tight_layout()
plt.show()

display_md("### Sampled Launch / Occupancy Summary")
display(sampled_summary(sampled_launch_occ))


## SM / Compute vs Memory


In [ ]:
compute_summary = metric_summaries(["ComputeWorkloadAnalysis"])
memory_summary = metric_summaries(["MemoryWorkloadAnalysis"])
sol_summary = metric_summaries(["SpeedOfLight"])

compute_focus = compute_summary[
    compute_summary["metric_name"].isin(
        [
            "SM Busy",
            "Issue Slots Busy",
            "Executed Ipc Active",
            "Executed Ipc Elapsed",
            "Issued Ipc Active",
        ]
    )
].reset_index(drop=True)
memory_focus = memory_summary[
    memory_summary["metric_name"].isin(
        [
            "Memory Throughput",
            "Mem Busy",
            "Mem Pipes Busy",
            "Max Bandwidth",
            "L1/TEX Hit Rate",
            "L2 Hit Rate",
        ]
    )
].reset_index(drop=True)
sol_focus = sol_summary[
    sol_summary["metric_name"].isin(
        [
            "Compute (SM) Throughput",
            "Memory Throughput",
            "DRAM Throughput",
            "L1/TEX Cache Throughput",
            "L2 Cache Throughput",
            "Duration",
        ]
    )
].reset_index(drop=True)

display_md("### Compute Workload Analysis")
display(compute_focus)

display_md("### Memory Workload Analysis")
display(memory_focus)

display_md("### GPU Speed Of Light Throughput")
display(sol_focus)


In [ ]:
throughput_pct = pd.Series(
    {
        "Compute (SM) Throughput": metric_value(sol_summary, "SpeedOfLight", "Compute (SM) Throughput"),
        "Memory Throughput": metric_value(sol_summary, "SpeedOfLight", "Memory Throughput"),
        "DRAM Throughput": metric_value(sol_summary, "SpeedOfLight", "DRAM Throughput"),
        "L1/TEX Cache Throughput": metric_value(sol_summary, "SpeedOfLight", "L1/TEX Cache Throughput"),
        "L2 Cache Throughput": metric_value(sol_summary, "SpeedOfLight", "L2 Cache Throughput"),
    }
)
busy_and_hit_rates = pd.Series(
    {
        "SM Busy": metric_value(compute_summary, "ComputeWorkloadAnalysis", "SM Busy"),
        "Issue Slots Busy": metric_value(compute_summary, "ComputeWorkloadAnalysis", "Issue Slots Busy"),
        "Mem Busy": metric_value(memory_summary, "MemoryWorkloadAnalysis", "Mem Busy"),
        "Mem Pipes Busy": metric_value(memory_summary, "MemoryWorkloadAnalysis", "Mem Pipes Busy"),
        "L1/TEX Hit Rate": metric_value(memory_summary, "MemoryWorkloadAnalysis", "L1/TEX Hit Rate"),
        "L2 Hit Rate": metric_value(memory_summary, "MemoryWorkloadAnalysis", "L2 Hit Rate"),
    }
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_bar(axes[0], throughput_pct, "Speed-of-Light Throughput Signals", "Percent")
plot_bar(axes[1], busy_and_hit_rates, "Busy and Hit-Rate Signals", "Percent")
plt.tight_layout()
plt.show()


## GPU and Memory Workload Distribution


In [ ]:
workload_summary = metric_summaries(["WorkloadDistribution"])
workload_focus = summary_rows(workload_summary, ["Active Cycles", "Elapsed Cycles"])

display_md("### WorkloadDistribution Summary")
display(workload_focus)

workload_resources = ["SM", "SMSP", "L1", "L2", "DRAM"]
workload_rows = []
workload_pairs = []

for resource in workload_resources:
    active_name = f"Average {resource} Active Cycles"
    elapsed_name = f"Total {resource} Elapsed Cycles"
    active_value = metric_value(workload_summary, "WorkloadDistribution", active_name)
    elapsed_value = metric_value(workload_summary, "WorkloadDistribution", elapsed_name)
    if pd.isna(active_value) and pd.isna(elapsed_value):
        continue
    workload_rows.append(
        {
            "resource": resource,
            "active_metric": active_name,
            "avg_active_cycles": active_value,
            "elapsed_metric": elapsed_name,
            "avg_elapsed_cycles": elapsed_value,
            "active_over_elapsed_pct": ratio_pct(active_value, elapsed_value),
        }
    )
    if not pd.isna(active_value):
        workload_pairs.append(("WorkloadDistribution", active_name))
    if not pd.isna(elapsed_value):
        workload_pairs.append(("WorkloadDistribution", elapsed_name))

workload_resource_df = pd.DataFrame(workload_rows)

display_md("### Resource-Level Readout")
display(workload_resource_df)


In [ ]:
workload_util_series = (
    workload_resource_df.set_index("resource")["active_over_elapsed_pct"].dropna()
    if not workload_resource_df.empty
    else pd.Series(dtype=float)
)
workload_active_series = (
    workload_resource_df.set_index("resource")["avg_active_cycles"].dropna()
    if not workload_resource_df.empty
    else pd.Series(dtype=float)
)

sampled_workload = sampled_metrics(workload_pairs)
display_md("### Sampled WorkloadDistribution Summary")
display(sampled_summary(sampled_workload))

sampled_workload_pivot = sampled_workload.pivot_table(
    index="sample_rank",
    columns="metric_name",
    values="metric_value_num",
    aggfunc="mean",
)
sampled_workload_ratio_df = pd.DataFrame(index=sampled_workload_pivot.index)

for resource in workload_resources:
    active_name = f"Average {resource} Active Cycles"
    elapsed_name = f"Total {resource} Elapsed Cycles"
    if active_name in sampled_workload_pivot.columns and elapsed_name in sampled_workload_pivot.columns:
        sampled_workload_ratio_df[resource] = 100.0 * sampled_workload_pivot[active_name] / sampled_workload_pivot[elapsed_name].replace(0, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
if not workload_util_series.empty:
    plot_bar(axes[0], workload_util_series, "WorkloadDistribution Active / Elapsed Ratio", "Percent")
else:
    axes[0].set_visible(False)

if not workload_active_series.empty:
    plot_bar(axes[1], workload_active_series, "Average Active Cycles by Resource", "Cycles")
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()

if sampled_workload_ratio_df.empty:
    display_md("""### Sampled WorkloadDistribution Chart
No sampled active/elapsed pairs were available for this bundle.""")
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    for resource in sampled_workload_ratio_df.columns:
        sampled_workload_ratio_df[resource].plot(ax=ax, label=resource)
    ax.set_title("Sampled WorkloadDistribution Active / Elapsed Ratio")
    ax.set_xlabel("Sample Rank")
    ax.set_ylabel("Percent")
    ax.legend(title="Resource")
    plt.tight_layout()
    plt.show()


## Scheduler and Warp Behavior


In [ ]:
scheduler_summary = metric_summaries(["SchedulerStats"])
warp_summary = metric_summaries(["WarpStateStats"])

scheduler_focus = scheduler_summary[
    scheduler_summary["metric_name"].isin(
        [
            "Active Warps Per Scheduler",
            "Eligible Warps Per Scheduler",
            "Issued Warp Per Scheduler",
            "No Eligible",
            "One or More Eligible",
        ]
    )
].reset_index(drop=True)
warp_focus = warp_summary[
    warp_summary["metric_name"].isin(
        [
            "Avg. Active Threads Per Warp",
            "Avg. Not Predicated Off Threads Per Warp",
            "Warp Cycles Per Executed Instruction",
            "Warp Cycles Per Issued Instruction",
        ]
    )
].reset_index(drop=True)

display_md("### Scheduler Statistics")
display(scheduler_focus)

display_md("### Warp State Statistics")
display(warp_focus)


In [ ]:
scheduler_pct = pd.Series(
    {
        "No Eligible": metric_value(scheduler_summary, "SchedulerStats", "No Eligible"),
        "One or More Eligible": metric_value(scheduler_summary, "SchedulerStats", "One or More Eligible"),
    }
)
scheduler_warps = pd.Series(
    {
        "Active Warps / Scheduler": metric_value(scheduler_summary, "SchedulerStats", "Active Warps Per Scheduler"),
        "Eligible Warps / Scheduler": metric_value(scheduler_summary, "SchedulerStats", "Eligible Warps Per Scheduler"),
        "Issued Warp / Scheduler": metric_value(scheduler_summary, "SchedulerStats", "Issued Warp Per Scheduler"),
    }
)
warp_cycles = pd.Series(
    {
        "Warp Cycles / Executed Inst": metric_value(warp_summary, "WarpStateStats", "Warp Cycles Per Executed Instruction"),
        "Warp Cycles / Issued Inst": metric_value(warp_summary, "WarpStateStats", "Warp Cycles Per Issued Instruction"),
    }
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_bar(axes[0], scheduler_pct, "Scheduler Eligibility", "Percent")
plot_bar(axes[1], scheduler_warps, "Warps Visible to Each Scheduler", "Warp Count")
plot_bar(axes[2], warp_cycles, "Warp-State Cycle Signals", "Cycles")
plt.tight_layout()
plt.show()


## Nsight Guidance Cards


In [ ]:
rules_df = q(
    """
    SELECT
        section_id,
        rule_name,
        rule_type,
        MAX(rule_description) AS rule_description,
        MAX(estimated_speedup_type) AS estimated_speedup_type,
        ROUND(MAX(estimated_speedup), 2) AS max_estimated_speedup,
        COUNT(DISTINCT launch_id) AS launches_covered
    FROM section_metric_values
    WHERE rule_name IS NOT NULL AND TRIM(rule_name) != ''
    GROUP BY section_id, rule_name, rule_type
    ORDER BY launches_covered DESC, section_id, rule_name
    """
)
if matched_launches:
    rules_df["launch_coverage_pct"] = (100.0 * rules_df["launches_covered"] / matched_launches).round(2)

display(rules_df)

cards = []
for row in rules_df.head(8).itertuples(index=False):
    speedup = "not quantified"
    if not pd.isna(row.max_estimated_speedup):
        speedup_type = row.estimated_speedup_type or "estimated"
        speedup = f"{row.max_estimated_speedup:0.2f} ({speedup_type})"
    description = row.rule_description if row.rule_description else "No structured rule description was stored for this row."
    coverage = f"{row.launches_covered} launches"
    if "launch_coverage_pct" in rules_df.columns:
        coverage += f" ({getattr(row, 'launch_coverage_pct', np.nan):0.2f}% of matched launches)"
    cards.append(
        dedent(
            f"""
            ### {row.rule_name} [{row.section_id}]
            - Rule type: `{row.rule_type or 'n/a'}`
            - Launch coverage: `{coverage}`
            - Max estimated speedup: `{speedup}`
            - Description: {description}
            """
        ).strip()
    )

display_md("\n\n".join(cards) if cards else "No structured rules were captured in the section rows.")


## Per-launch Distributions


In [ ]:
distribution_pairs = [
    ("LaunchStats", "Waves Per SM"),
    ("Occupancy", "Achieved Occupancy"),
    ("SchedulerStats", "Eligible Warps Per Scheduler"),
    ("ComputeWorkloadAnalysis", "SM Busy"),
    ("MemoryWorkloadAnalysis", "Memory Throughput"),
    ("SpeedOfLight", "Duration"),
    ("WorkloadDistribution", "Average DRAM Active Cycles"),
]
sampled_distribution_df = sampled_metrics(distribution_pairs)

display_md("### Sampled Distribution Summary")
display(sampled_summary(sampled_distribution_df))

pivot = sampled_distribution_df.pivot_table(
    index="sample_rank",
    columns=["section_id", "metric_name"],
    values="metric_value_num",
    aggfunc="mean",
)

ncols = 2
nrows = int(np.ceil(len(distribution_pairs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows), sharex=True)
axes = np.atleast_1d(axes).reshape(nrows, ncols)
for ax, pair in zip(axes.flat, distribution_pairs):
    if pair in pivot.columns:
        pivot[pair].plot(ax=ax, color="#4C72B0")
        ax.set_title(f"{pair[0]}: {pair[1]}")
        ax.set_xlabel("Sample Rank")
        ax.set_ylabel("Value")
    else:
        ax.set_visible(False)
for ax in axes.flat[len(distribution_pairs):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()


## Guided Interpretation


In [ ]:
all_summary = metric_summaries(
    [
        "LaunchStats",
        "Occupancy",
        "SchedulerStats",
        "WarpStateStats",
        "ComputeWorkloadAnalysis",
        "MemoryWorkloadAnalysis",
        "SpeedOfLight",
        "WorkloadDistribution",
    ]
)

waves = metric_value(all_summary, "LaunchStats", "Waves Per SM")
achieved_occ = metric_value(all_summary, "Occupancy", "Achieved Occupancy")
theoretical_occ = metric_value(all_summary, "Occupancy", "Theoretical Occupancy")
eligible_warps = metric_value(all_summary, "SchedulerStats", "Eligible Warps Per Scheduler")
no_eligible = metric_value(all_summary, "SchedulerStats", "No Eligible")
sm_busy = metric_value(all_summary, "ComputeWorkloadAnalysis", "SM Busy")
mem_busy = metric_value(all_summary, "MemoryWorkloadAnalysis", "Mem Busy")
compute_throughput = metric_value(all_summary, "SpeedOfLight", "Compute (SM) Throughput")
memory_throughput = metric_value(all_summary, "SpeedOfLight", "Memory Throughput")
l2_hit = metric_value(all_summary, "MemoryWorkloadAnalysis", "L2 Hit Rate")
duration_us = metric_value(all_summary, "SpeedOfLight", "Duration")

workload_ratio_series = (
    workload_resource_df.set_index("resource")["active_over_elapsed_pct"].dropna()
    if "workload_resource_df" in globals() and not workload_resource_df.empty
    else pd.Series(dtype=float)
)
if workload_ratio_series.empty:
    workload_text = "WorkloadDistribution metrics were not available for this bundle."
else:
    workload_top_resource = workload_ratio_series.sort_values(ascending=False).index[0]
    workload_top_ratio = workload_ratio_series.max()
    workload_pairs_text = ", ".join(
        f"{resource} {value:0.2f}%" for resource, value in workload_ratio_series.items()
    )
    workload_text = (
        f"WorkloadDistribution active/elapsed ratios remain low across the hierarchy ({workload_pairs_text}); "
        f"the highest observed ratio is `{workload_top_resource}` at `{pct(workload_top_ratio)}`."
    )

top_rule = rules_df.iloc[0] if not rules_df.empty else None
top_rule_text = "No structured rule/advice rows were captured."
if top_rule is not None:
    speedup_text = "not quantified"
    if not pd.isna(top_rule.max_estimated_speedup):
        speedup_text = f"{top_rule.max_estimated_speedup:0.2f} ({top_rule.estimated_speedup_type or 'estimated'})"
    top_rule_text = (
        f"Top repeated guidance card: `{top_rule.rule_name}` in `{top_rule.section_id}` covering "
        f"{int(top_rule.launches_covered)} launches with max estimated speedup `{speedup_text}`."
    )

display_md(
    f"""
    ### Structured Narrative
    - Launch scale is modest for the device: average `Waves Per SM` is `{num(waves)}`, so this kernel does not keep many full waves resident at once.
    - Occupancy stays low relative to its own ceiling: achieved occupancy is `{pct(achieved_occ)}` versus theoretical occupancy `{pct(theoretical_occ)}`.
    - Scheduler pressure remains weak: eligible warps per scheduler average `{num(eligible_warps)}` and schedulers report `No Eligible` `{pct(no_eligible)}` of the time.
    - Compute and memory pipelines both look active but not saturated: `SM Busy` is `{pct(sm_busy)}`, `Mem Busy` is `{pct(mem_busy)}`, `Compute (SM) Throughput` is `{pct(compute_throughput)}`, and `Memory Throughput` is `{pct(memory_throughput)}`.
    - WorkloadDistribution adds hierarchy context rather than overturning the main story: {workload_text}
    - Memory locality is mixed rather than catastrophic: `L2 Hit Rate` averages `{pct(l2_hit)}` while the average sampled kernel duration is `{num(duration_us)}` microseconds.
    - {top_rule_text}

    This combination still reads more like an underfilled / low-eligibility kernel than a kernel that is simply pinned against a single throughput ceiling. The added workload-distribution section suggests activity is spread across the hierarchy, but it does not reveal a single saturated tier that would displace the launch, occupancy, scheduler, and speed-of-light evidence.
    """
)


## Coverage Gaps


In [ ]:
coverage_gaps = pd.DataFrame(
    [
        {
            "area": "Core structured views",
            "status": "covered",
            "details": "Launch, Occupancy, Scheduler, Warp State, Compute Workload, Memory Workload, Speed of Light, and WorkloadDistribution are all imported and surfaced in this notebook.",
        },
        {
            "area": "Nsight guidance cards",
            "status": "covered",
            "details": "Structured rule rows are shown directly from section imports rather than only from embedded report strings.",
        },
        {
            "area": "GPU and Memory Workload Distribution",
            "status": "covered",
            "details": "The default core profile now imports WorkloadDistribution, and the notebook surfaces raw cycle metrics, derived active/elapsed ratios, and sampled-launch charts.",
        },
        {
            "area": "Source / instruction views",
            "status": "deferred",
            "details": "Not part of the default capture profile and still outside the current notebook scope.",
        },
        {
            "area": "Full Nsight Compute UI parity",
            "status": "deferred",
            "details": "The notebook is now a guided structured review surface, but it still does not attempt full roofline/source/SASS parity.",
        },
    ]
)
display(coverage_gaps)

partial_sections = section_attempts[~section_attempts["status"].isin(SUCCESS_STATUSES)]
if partial_sections.empty:
    display_md("""### Core Section Status
All eight core sections completed cleanly for this bundle.""")
else:
    display_md("""### Core Section Status
Some sections remain partial or missing:""")
    display(partial_sections)
